This notebook describes the procedure for a redshift measurement for a single spectrum

# Spectrum import

We first need to import spectrum of which redshift will be measured.

The format of the spectrum should be (4xn_pixel) array with

- 1st array: wavelength (in ascending order)

- 2nd array: flux

- 3rd array: flux uncertainty (standard deviation)

- 4th array: one for pixel used in cross-correlation, zero for one not used in cross-correlation

RVSNUpy can measure the redshifts of any spectrum that has the above format.

Current RVSNUpy provide a function that converts a fits file for MMT/Hectospec and SDSS into the above format.

In [2]:
# Read sdss spectrum
from RVSNUpy.spectrum_import import sdss_fits
sdss_spec = sdss_fits('spectra/sdss_spec1.fits')

# Read MMT/Hectospec spectrum
from RVSNUpy.spectrum_import import MMT_raw
mmt_spec = MMT_raw('spectra/mmt_spec1.fits')

# Read flux-calibrated MMT/Hectospec spectrum
from RVSNUpy.spectrum_import import MMT_flux
mmt_spec = MMT_flux('spectra/mmt_flux_calibrated_spec1.fits')

/Users/gimtaewan/opt/anaconda3/lib/python3.8/site-packages/RVSNUpy/spectrum_import.py:118: RuntimeWarning: invalid value encountered in sqrt
  uncertainty = (1/np.sqrt(uncertainty))


** For other spectra, users should manually convert a spectrum to above format.

** If you could share the code to convert a spectrum file to the format, I can add that to RVSNUpy

# Initialize RVSNUpy

## Import a template set

To use RVSNUpy, users should import templates used for redshift measurements.

Current RVSNUpy provides a sdss-based galaxy template set.

In [2]:
from RVSNUpy.template import sdss_galaxy_templates
gal_temps = sdss_galaxy_templates("vacuum")

sdss_galaxy_templates() takes an input for wavelength system and returns the template set
- "vacuum": vacuum wavelength system
- "air": air wavelength system

Users need to specify the wavelength system of RVSNUpy's sdss-based galaxy template set depending on their spectrum's wavelength system.

** Depending on the purpose, users can use other templates; to construct a new template set, please refer to template_construction.ipynb

If the template covers optical range (~3900-8000 ${\rm \AA}$) in the rest frame, their radial velocity can be calibrated to zero with 'zp_calib'

In [3]:
from RVSNUpy.template import zp_calib
gal_temps = zp_calib(gal_temps)

## Initialize rvm

'rvm' is the main part of RVSNUpy, which actually measures redshifts.
Most measurements and analysis will be performed in 'rvm'.

Using rvm starts with initilizing rvm.
This process takes the template set as an input and shifts the templates across the various redshift for initial cross-correlation (see Section 3.3 of Kim et a. 2025)

In [ ]:
from RVSNUpy.rvm import rvm
measure = rvm(gal_temps)

Other than the template set, the following parameters are taken as inputs:

- z_range ([float, float], default: [0,1]): The redshift range within which the spectrum is searched for a redshifting solution. Default is [0, 1].
- temp_apodization_size (float, default: 0.05): The apodization size used for normalizing the template spectrum. See Section 3.2 of Kim et al. (2025) for details.
- temp_knots_bin (float, default: 100): The bin size for the B-spline knots in ${\rm \AA}$ used in continuum normalization of the templates. See Section 3.2 of Kim et al. (2025). For templates with broad emission lines, increasing this value is recommended.
- temp_line_thres (int, default: 3): The threshold (in sigma) used to identify and mask emission or absorption lines in the template spectrum.
- temp_resolution (int, default: 3): The spectral resolution of the template.
- n_jobs (int, default: -1): The number of CPU cores to use. Set to -1 to use all available processors.

# Single measurement

'rvm' provides a method called 'z_single' to measure the redshift of a single spectrum.

The 'z_single' method takes a spectrum in the form of 4 × n_pixel array as input and returns the redshift measurement results for all templates in a pandas.DataFrame. The output DataFrame includes the following columns:
- template_name: Name of the template used for the measurement.
- z: Estimated redshift based on the corresponding template.
- zerr: Uncertainty in the estimated redshift.
- r: Significance of the redshift measurement. Values of r > 5 generally indicate a reliable measurement, while r > 3 typically suggest a good measurement.
- chi_eff: Effective (reduced) chi-square comparing the input spectrum and the template. If the template is emission line dominated, it indicate the chi-square based on line fittings (see Section 3.4 of Kim et al. 2025). Values close to 1 indicate a good match, while chi_eff > 4 typically implies a poor match between the template and the input spectrum.
- note: Marks the best redshift measurement among all templates.

In [7]:
df=measure.z_single(sdss_spec)
print(df)

         template_name         z      zerr          r   chi_eff  note
0     Late_type_galaxy  0.167850  0.000070  15.120880  1.161129      
1    Early_type_galaxy  0.167847  0.000066   9.567400  1.268727      
2  Luminous_red_galaxy  0.167831  0.000068   9.204723  1.153615  best


In addition to the spectrum, the following parameters can be provided to the z_single method:
- output (str, default: 'all'): If set to 'all', z_single returns the redshift measurement results for all templates. If set to 'best', only the best redshift measurement result is returned. 
- prior (str, default: 'abs'): If 'abs', absorption-line-dominated templates are preferred when selecting the best redshift measurement. If 'em', emission-line-dominated templates are preferred. 
- spectrum_range (list, [λ₁, λ₂], default: None): The wavelength range in ${\rm \AA}$ used for redshift measurement. If None, the entire wavelength range of the input spectrum is used.
- mask (list of lists, [[λ₁, λ₂], [λ₃, λ₄], ...], default: None): Wavelength regions to be excluded from the redshift measurement. If None, no masking is applied.
- sn_continuum (float, default: 0.5): Minimum signal-to-noise ratio required at the edges of the spectrum. If the S/N at either end is below this threshold, that portion of the spectrum is excluded from analysis.
- window_continuum (int, default: 100): The window size in ${\rm \AA}$ removed from the edges of the spectrum when the signal-to-noise is below the sn_continuum threshold.
- resolution (float, default: 3): The spectral resolution of the input spectrum.
- chi_thres (float, default: 4): Redshift measurement results with a reduced chi-square (chi_eff) greater than this threshold are excluded when determining the best measurement. See Section 3.5 of Kim et al. (2025).
- r_thres (float, default: 5): Redshift measurement results with a significance (r) below this threshold are excluded when determining the best measurement.
- knots_bin (float, default: 100): The bin size (in ${\rm \AA}$) for B-spline knots used in continuum normalization of the input spectrum. See Section 3.2 of Kim et al. (2025). For spectra with broad emission lines, increasing this value is recommended.
- line_thres (float, default: 3): The threshold (in sigma) used to identify and mask emission or absorption lines in the input spectrum.
- apodization_size (float, default: 0.05): The apodization size used in continuum normalization to smoothly taper the edge of the input spectrum. See Section 3.2 of Kim et al. (2025).
- line_fit (bool, default: True): If True, rvm performs line fitting when using emission-line-dominated templates If False, redshift is determined purely by cross-correlation, as done for absorption-line-dominated templates.
- em_lines (list of float or list of lists): A list of emission-line wavelengths (in ${\rm \AA}$) used for line fitting. See Section 3.2 of Kim et al. (2025). Lines grouped in a sublist are fit simultaneously. 

# Multiplie redshift measurement

'rvm' also provides two methods for measuring redshifts of multiple spectra.

(1) 'z_speclist': measures the redshifts of spectra cotained in the list.

(2) 'z_multi': measures the redshifts of spectra indicated by the pathes in the list

## z_speclist

'z_speclist' takes the list of spectra as input and returns the best redshift measurement for each spectrum. All parameters for z_single can be used for 'z_speclist'

In [25]:
# list of spectrum
import glob
SpecList = [sdss_fits(f) for f in glob.glob('spectra/sdss_spec*.fits')]

# measured with 'z_speclist'
df = measure.z_speclist(SpecList)
print(df)

         best_template         z      zerr          r   chi_eff
0    Early_type_galaxy  0.058758  0.000071   7.998151  1.120633
1  Luminous_red_galaxy  0.133113  0.000046  17.251406  1.009059
2  Luminous_red_galaxy  0.167831  0.000068   9.204724  1.153615
3    Early_type_galaxy  0.135603  0.000037  13.243608  1.181444
4    Early_type_galaxy  0.030239  0.000068  10.046979  1.097027
5  Luminous_red_galaxy  0.391101  0.000102  11.839429  1.132095
6  Luminous_red_galaxy  0.143029  0.000057  11.118220  1.096215
7    Early_type_galaxy  0.071963  0.000082   7.795314  1.217573
8  Luminous_red_galaxy  0.260901  0.000075  17.043510  1.059343
9  Luminous_red_galaxy  0.167831  0.000068   9.204724  1.153615


## z_mutli

z_multi takes a list of spectrum file paths and a user-defined function that converts each file into a 4 × n_pixel array. It create the z_result.txt file cotaning the best redshift measurement result for each spectrum file

In [29]:
SpecFiles = [f for f in glob.glob('spectra/sdss_spec*.fits')]
measure.z_multi(SpecFiles, sdss_fits)

importing spectra for the 1st chunk...


done
measuring redshifts for the 1st chunk...


done


Other parameters for 'z_multi' are
- chunk (int, default: 5000): the number of spectra imported at the same time
- direcotry (str, default: 'z_result'): the directory where z_result.txt will be save
- all other parameters for 'z_single'